In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings("ignore")
from sklearn.feature_extraction.text import CountVectorizer , TfidfVectorizer

In [3]:
df = pd.read_csv("cleaned_books.csv")

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      50 non-null     int64
 1   title   50 non-null     str  
 2   tags    50 non-null     str  
dtypes: int64(1), str(2)
memory usage: 12.9 KB


In [5]:
tfidf = TfidfVectorizer(max_features = 10000, stop_words='english')
vector = tfidf.fit_transform(df['tags']).toarray()

In [6]:
vector.shape

(50, 723)

In [7]:
similarity = cosine_similarity(vector)
similarity

array([[1.        , 0.01599978, 0.04317577, ..., 0.02997851, 0.        ,
        0.        ],
       [0.01599978, 1.        , 0.16540496, ..., 0.        , 0.        ,
        0.02898254],
       [0.04317577, 0.16540496, 1.        , ..., 0.06621561, 0.        ,
        0.        ],
       ...,
       [0.02997851, 0.        , 0.06621561, ..., 1.        , 0.08093037,
        0.03086199],
       [0.        , 0.        , 0.        , ..., 0.08093037, 1.        ,
        0.10774772],
       [0.        , 0.02898254, 0.        , ..., 0.03086199, 0.10774772,
        1.        ]], shape=(50, 50))

In [8]:
similarity.shape

(50, 50)

In [9]:
def get_name_by_index(i):
    if i < len(df) and i>0:
        return df.loc[i,'title']
    else:
        return''

In [10]:
a = get_name_by_index(10)
a

'The Martian'

In [11]:
def get_index_from_name(name):
    # Normalize user input: lowercase it and strip out all spaces and hyphens
    clean_user_name = name.strip().lower().replace(' ', '').replace('-', '')
    
    # Vectorized pandas match: normalize the dataframe column for comparison
    match = df[df['title'].str.lower().str.replace(' ', '').str.replace('-', '') == clean_user_name]
    if not match.empty:
        return match.index[0]
    return -1

In [12]:
get_index_from_name("the hobbit")

1

In [13]:
name = input("Enter a book you've read:")
index = get_index_from_name(name)

if index != -1:
    # Fetch similarity rankings. enumerate() pairs each score with its original book index
    similarity_indexes = list(enumerate(similarity[index]))

    # The very first item in this sorted list will always be the book itself (score 1.0)
    similarity_indexes = sorted(similarity_indexes, key=lambda x: x[1], reverse=True)
    
    print(f"\nBecause you read '{df.loc[index, 'title']}':")
    print("You might also like:")
    
    # Loop to print top 5 recommendations (skipping index 0, the book itself)
    for i in range(1, 6):
        book_idx = similarity_indexes[i][0]
        print(i, ":", get_name_by_index(book_idx))
else:
    print("Book Not Found")

Enter a book you've read: the hobbit



Because you read 'The Hobbit':
You might also like:
1 : The Alchemist
2 : The Lord of the Rings
3 : Me Before You
4 : Brave New World
5 : The Handmaid's Tale


In [14]:
#Lets export similarity
import pickle as pkl
pkl.dump(similarity, open('similarity.pkl', 'wb'))